# Stage 4 — Intervention analysis: did ablating the latent direction reduce output bias?

Loads `results/intervention_results/`, joins to baseline `results/logit_scores/`, and computes the before-vs-after bias deltas per (model, attribute, benchmark, prompt_mode, method).

Three things we want to read off:
1. **Headline delta**: bias_baseline − bias_intervened. Positive = ablation reduced bias (causal evidence).
2. **Mechanism specificity**: ablating *gender* should drop gender-CrowS pairs more than race-color pairs.
3. **Method robustness**: INLP and LEACE should agree on direction and rough magnitude.

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive; drive.mount('/content/drive')
    %cd /content/llm-bias-eval
    if not os.path.lexists('results'):
        os.symlink('/content/drive/MyDrive/llm-bias-eval/results', 'results')

In [ ]:
from pathlib import Path
import pandas as pd

from biaseval.analysis.aggregate_results import (
    aggregate_logit_results, aggregate_intervention_results,
)

RESULTS = Path('results')
TABLES = Path('results/tables'); TABLES.mkdir(parents=True, exist_ok=True)

logit_df = aggregate_logit_results(RESULTS)
intv_df  = aggregate_intervention_results(RESULTS)
print(f'logit rows={len(logit_df)}, intervention rows={len(intv_df)}')

## 1. Sanity-check audit

Filter out any (model, attribute, method) cells where the intervention failed its sanity checks (probe didn't get nullified, or perplexity blew up). These shouldn't be interpreted as bias-mitigation evidence.

In [ ]:
if not intv_df.empty:
    audit = (intv_df[['model_id', 'attribute', 'method', 'probe_acc_post',
                       'probe_acc_passed', 'perplexity_ratio', 'perplexity_passed']]
             .drop_duplicates())
    print('Failed sanity checks:')
    print(audit[~(audit['probe_acc_passed'].fillna(False) & audit['perplexity_passed'].fillna(False))])
    audit.to_csv(TABLES / 'intervention_sanity.csv', index=False)

## 2. Headline before/after table

For each (model, benchmark, prompt_mode, attribute, method): bias_baseline (no intervention) vs bias_intervened, with delta. Headline metric per benchmark: CrowS overall stereo-win-rate, StereoSet SS.

In [ ]:
HEADLINE = {'crows_pairs': 'overall', 'stereoset': 'overall_SS'}

if intv_df.empty:
    delta_df = pd.DataFrame()
else:
    intv_keep = intv_df[intv_df.apply(lambda r: r['metric'] == HEADLINE.get(r['benchmark']), axis=1)].copy()
    intv_keep = intv_keep.rename(columns={'value': 'bias_intervened'})

    base = logit_df[logit_df.apply(lambda r: r['metric'] == HEADLINE.get(r['benchmark']), axis=1)].copy()
    base = base.rename(columns={'value': 'bias_baseline'})

    delta_df = intv_keep.merge(
        base[['model_id', 'benchmark', 'prompt_mode', 'bias_baseline']],
        on=['model_id', 'benchmark', 'prompt_mode'], how='left',
    )
    delta_df['bias_delta'] = delta_df['bias_baseline'] - delta_df['bias_intervened']
    cols = ['family', 'model_id', 'variant', 'benchmark', 'prompt_mode',
            'attribute', 'method', 'bias_baseline', 'bias_intervened', 'bias_delta']
    delta_df = delta_df[cols].sort_values(cols[:6])
    delta_df.to_csv(TABLES / 'intervention_delta.csv', index=False)
delta_df.head(20)

## 3. Headline figure — bias delta per (family, attribute, method)

Group by family × attribute × method, average across models/prompt_modes/benchmarks. Positive bars = ablating that attribute's latent direction reduced output bias (causal evidence).

In [ ]:
import matplotlib.pyplot as plt
FIGURES = Path('figures'); FIGURES.mkdir(exist_ok=True)

if not delta_df.empty:
    pivot = (delta_df.groupby(['family', 'attribute', 'method'])['bias_delta']
             .mean().unstack('method'))
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind='bar', ax=ax)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_ylabel('Δ bias score (baseline − intervened)')
    ax.set_title('Output bias reduction after ablating latent attribute direction')
    plt.tight_layout()
    plt.savefig(FIGURES / 'intervention_delta.png', dpi=150)
    plt.show()

## 4. Mechanism check — does ablating gender selectively reduce gender-CrowS pairs?

Per-bias-type breakdown of intervention effect. If the ablation is causally specific, ablating *gender* should drop the **gender** category of CrowS-Pairs more than e.g. **race-color** or **religion**.

In [ ]:
if not intv_df.empty:
    # CrowS-Pairs has per-bias-type metrics in its summary (e.g. 'gender', 'race-color', ...)
    bias_types = ['gender', 'race-color', 'religion', 'age', 'disability',
                  'physical-appearance', 'sexual-orientation', 'nationality', 'socioeconomic']
    crows_intv = intv_df[(intv_df['benchmark'] == 'crows_pairs') & intv_df['metric'].isin(bias_types)]
    crows_base = logit_df[(logit_df['benchmark'] == 'crows_pairs') & logit_df['metric'].isin(bias_types)]
    crows_intv = crows_intv.rename(columns={'value': 'intervened', 'metric': 'bias_type'})
    crows_base = crows_base.rename(columns={'value': 'baseline', 'metric': 'bias_type'})
    mech = crows_intv.merge(
        crows_base[['model_id', 'prompt_mode', 'bias_type', 'baseline']],
        on=['model_id', 'prompt_mode', 'bias_type'], how='left',
    )
    mech['delta'] = mech['baseline'] - mech['intervened']
    mech_summary = (mech.groupby(['attribute', 'method', 'bias_type'])['delta']
                    .mean().unstack('bias_type'))
    mech_summary.to_csv(TABLES / 'intervention_mechanism_check.csv')
    print(mech_summary.round(2))

## 5. Variant × intervention interaction (regression)

Does the intervention reduce bias *more* for instruct models than base? If yes, evidence that alignment was relying on the latent direction.

In [ ]:
if not delta_df.empty and len(delta_df) > 10:
    import statsmodels.formula.api as smf
    m = smf.ols(
        "bias_delta ~ C(variant) + C(method) + C(attribute) + C(benchmark) + C(prompt_mode)",
        data=delta_df,
    ).fit(cov_type='cluster', cov_kwds={'groups': delta_df['model_id'].values})
    print(m.summary())